# Experiment: ablación de etiquetas BI-RADS (VinDr-Mammo)

**Pregunta:** ¿el techo actual (ROC-AUC ~0.75, PR-AUC ~0.30) viene de meter BI-RADS 3 en la clase positiva?

**Setup fijo (no se toca `src/`):** backbone `resized_448_swint`, mismos preprocess, augmentations, batch, optimizador, scheduler y épocas. Solo cambia la definición binaria de `cls`.

**Tres formulaciones:**

| ID | Negativo | Positivo | BI-RADS 3 |
| --- | --- | --- | --- |
| A | 1–2 | 3–5 | positivo (setup actual) |
| B | 1–2 | 4–5 | **excluido** de train/val/test |
| C | 1–3 | 4–5 | negativo |

**Criterio de éxito:**

- Si **B mejora mucho** respecto de A → BI-RADS 3 está bajando el techo.
- Si **C mejora pero cambia mucho la prevalencia** → reportarlo; no es comparable a ojo con PR-AUC crudo.
- Si los tres siguen cerca de ROC-AUC ~0.75 → el cuello no es BI-RADS 3 (señal visual, domain shift, resolución, arquitectura o formulación por vista/estudio).

**Umbral:** Youden sobre **validación**. Nunca se elige umbral con test.

**Splits:** se reutiliza `splits/dataset_splits.json` (agrupado por paciente). Después se reetiqueta. Si existe `study_id`, el bootstrap y el chequeo de leakage usan estudio; si no, `patient_id`.


In [ ]:
from pathlib import Path

# En Colab, para forzar un src fresco (esta consigna no modifica src):
# !rm -rf ./src && git clone https://github.com/emiliodelgadouy/TesisV4_src src

if not Path("./src/notebook.py").is_file():
    !git clone https://github.com/emiliodelgadouy/TesisV4_src src
else:
    print(f"Reusando {Path('./src').resolve()} (no se clona ni se modifica src)")

from src.notebook import configure_notebook

configure_notebook(autoreload=True)
from src.notebook_api import *


## Configuración

Mismo pipeline que `multirun.ipynb`, recortado a **un** backbone y **un** canvas. `EVALUATION.EXPORT_PREDICTIONS=True` guarda CSVs alineados para métricas extra y bootstrap agrupado (el bootstrap de `src` es por imagen; acá se ignora y se calcula agrupado en el notebook).

`REGENERATE_SPLITS=False`: A/B/C comparten la misma asignación paciente → split. Solo cambia `cls` (y el dropeo de BI-RADS 3 en B).


In [ ]:
from pathlib import Path

GENERAL = {
    "RANDOM_SEED": 42,
    "REDUCED_DATASET": False,
    "USE_CLAHE": True,
    "TARGET_MODE": "birads",
    "VALIDATION_SPLIT_RATIO": 0.20,
    "PROBABILITY_THRESHOLD": 0.5,
    "NO_FINDING_TO_FINDING_RATIO": 3,
    "METRIC_TO_MAXIMIZE": "pr_auc",
    "BATCH_SIZE": 32,
    "USE_CUSTOM_BATCH_SIZE": True,
    "CACHE_DATASET": True,
    "REGENERATE_SPLITS": False,
    "BIRADS_LABEL_SCHEME": "A",
}
TRAINING = {
    "EPOCHS_FROZEN_BACKBONE": 15,
    "EPOCHS_PARTIAL_BACKBONE": 25,
    "EPOCHS_FULL_FINETUNE": 15,
    "FOCAL_GAMMA": 2.0,
    "AGGRESSIVE_AUGMENTATION": True,
    "BACKBONE_TRAINABLE_FRACTION": 0.30,
    "EARLY_STOPPING_PATIENCE": 4,
    "REDUCE_LR_PATIENCE": 2,
}
ABMIL = {
    "BAG_GRID": (3, 3),
    "BAG_CANVAS_MODE": "resize",
    "BAG_KERAS_TILING": True,
    "BATCH_SIZE": 32,
    "ATTENTION_DIM": 128,
    "ATTENTION_GATED": True,
}
PATCH = {
    "POSITIVE": 1,
    "HARD_NEGATIVE": 0,
    "RANDOM_NEGATIVE": GENERAL["NO_FINDING_TO_FINDING_RATIO"],
    "RESIZE_TO_BAG_CANVAS": True,
}
PATCH_HARDNEG = {
    "POSITIVE": 1,
    "HARD_NEGATIVE": GENERAL["NO_FINDING_TO_FINDING_RATIO"] / 2,
    "RANDOM_NEGATIVE": GENERAL["NO_FINDING_TO_FINDING_RATIO"] / 2,
    "ALIGN_TO_BAG_GRID": True,
}
COMET = {
    "API_KEY": "SzBk2RvzTzZCQMpLNBh35mg1q",
    "PROJECT_NAME": "tesis_v3",
    "N_SAMPLE_IMAGES": 8,
}
MODELS = ["swint"]
RESIZED = {
    "INPUT_SIZES": [(448, 448)],
    "BATCH_SIZES": {448: 128, 672: 64, 896: 32, 1152: 16, 1536: 8, 2048: 4},
    "CACHE": {448: True, 672: True, 896: True, 1152: False, 1536: False, 2048: False},
}
EVALUATION = {
    "EXPORT_PREDICTIONS": True,
    "BOOTSTRAP_SAMPLES": 0,
}
CONFIG = {
    "GENERAL": GENERAL,
    "TRAINING": TRAINING,
    "ABMIL": ABMIL,
    "COMET": COMET,
    "MODELS": MODELS,
    "PATCH": PATCH,
    "PATCH_HARDNEG": PATCH_HARDNEG,
    "RESIZED": RESIZED,
    "EVALUATION": EVALUATION,
}

BACKBONE_NAME = "swint"
INPUT_SIZE = (448, 448)
RESULTS_DIR = Path("results")
RESULTS_CSV = RESULTS_DIR / "birads_label_ablation.csv"
RESULTS_MD = RESULTS_DIR / "birads_label_ablation.md"
BOOTSTRAP_REPS = 1000

set_random_seeds(CONFIG)
login_comet(CONFIG)
print("CONFIG listo: resized_448_swint, splits canónicos, predicciones exportadas")


## Dónde vive el pipeline (solo inspección; `src/` no se edita)

- **Labels:** `src/dataset/target.py` (`cls_from_birads`: 1–2 → 0, 3–5 → 1) y `src/dataset/repository.py` (`add_cls_column`). `DatasetPreparer` recorta por `TARGET_MODE=birads` y deja todas las filas 1–5.
- **Splits:** `src/dataset/splits.py`. Con `REGENERATE_SPLITS=False` carga IDs fijos (`patient_id`, `image_id`). Al regenerar, `StratifiedGroupKFold` agrupa por `patient_id`.
- **Dataloaders:** `src/dataset/provider.py` (`build_dataset_provider` / `build_splits`). Leen la columna `cls` ya resuelta.
- **Entrenamiento / umbral / métricas Keras:** `src/training/experiment.py` + `src/training/evaluator.py`. Youden y recall-target salen de **val**. El reporte Comet de test usa ese umbral.
- **Nombre Comet:** `ResizedModelBuilder.experiment_name` → `resized_448_swint`. En la celda de helpers se antepone `wip_{esquema}` **sin modificar el archivo de `src`**.


## Preparación del dataset y splits canónicos

`prepare_dataset` deja el `cls` oficial (A). `get_dataset_splits` fija qué imágenes van a train/val/test. Los esquemas B y C se aplican **después**, sobre copias de esos frames.


In [ ]:
ds = prepare_dataset(CONFIG)
train_raw, val_raw, test_raw = get_dataset_splits(CONFIG, ds)
print(ds.columns.tolist())
print("filas:", len(ds), "train/val/test:", len(train_raw), len(val_raw), len(test_raw))


## Esquemas de labels y chequeo de leakage

`LABEL_SCHEMES` es la opción de configuración del experimento. No se regenera el split: se reetiqueta (B además descarta BI-RADS 3). `focal_alpha` e `initial_bias` los recalcula el pipeline a partir del train de cada esquema.


In [ ]:
from src.dataset.target import parse_breast_birads
from src.model_builder.resized import ResizedModelBuilder
from src.training.mode import resized_size_label

LABEL_SCHEMES = {
    "A": {
        "key": "a_12_vs_35",
        "negative": (1, 2),
        "positive": (3, 4, 5),
        "drop": (),
        "title": "1-2 vs 3-5 (actual)",
    },
    "B": {
        "key": "b_12_vs_45",
        "negative": (1, 2),
        "positive": (4, 5),
        "drop": (3,),
        "title": "1-2 vs 4-5 (sin BI-RADS 3)",
    },
    "C": {
        "key": "c_123_vs_45",
        "negative": (1, 2, 3),
        "positive": (4, 5),
        "drop": (),
        "title": "1-3 vs 4-5 (BI-RADS 3 negativo)",
    },
}


def comet_prefix(scheme_id: str) -> str:
    return f"wip_{LABEL_SCHEMES[scheme_id]['key']}"


def _wip_experiment_name(cls, backbone_name, *, mode, input_size, suffix=None):
    del cls, mode
    name = f"resized_{resized_size_label(input_size)}_{backbone_name}"
    if suffix:
        return f"{suffix}_{name}"
    return name


ResizedModelBuilder.experiment_name = classmethod(_wip_experiment_name)


def birads_series(df):
    if "birads" in df.columns:
        return df["birads"]
    return parse_breast_birads(df["breast_birads"])


def resolve_group_column(*frames) -> str:
    for df in frames:
        if "study_id" in df.columns and df["study_id"].notna().any():
            return "study_id"
    return "patient_id"


def assert_no_group_leakage(train, val, test, group_col: str) -> None:
    sets = {
        name: set(frame[group_col].astype(str))
        for name, frame in (("train", train), ("val", val), ("test", test))
    }
    overlaps = {
        "train∩val": sets["train"] & sets["val"],
        "train∩test": sets["train"] & sets["test"],
        "val∩test": sets["val"] & sets["test"],
    }
    bad = {key: len(value) for key, value in overlaps.items() if value}
    if bad:
        raise ValueError(f"Leakage de {group_col}: {bad}")


def apply_label_scheme(train, val, test, scheme: dict, group_col: str):
    def _apply(frame):
        out = frame.copy()
        parsed = birads_series(out)
        out = out.assign(birads=parsed)
        if scheme["drop"]:
            out = out.loc[~out["birads"].isin(scheme["drop"])].copy()
        cls = pd.Series(pd.NA, index=out.index, dtype="Float32")
        cls = cls.mask(out["birads"].isin(scheme["negative"]), 0.0)
        cls = cls.mask(out["birads"].isin(scheme["positive"]), 1.0)
        out["cls"] = cls.astype("float32")
        out = out.loc[out["cls"].notna()].copy()
        return out.reset_index(drop=True)

    train_s, val_s, test_s = _apply(train), _apply(val), _apply(test)
    assert_no_group_leakage(train_s, val_s, test_s, group_col)
    return train_s, val_s, test_s


def split_count_row(frame, split_name: str) -> dict:
    n_pos = int((frame["cls"] == 1).sum())
    n_neg = int((frame["cls"] == 0).sum())
    n = n_pos + n_neg
    prevalence = float(n_pos / n) if n else float("nan")
    birads_counts = frame["birads"].value_counts(dropna=False).sort_index().to_dict()
    return {
        "split": split_name,
        "n": n,
        "n_pos": n_pos,
        "n_neg": n_neg,
        "prevalence": prevalence,
        "birads": birads_counts,
    }


GROUP_COL = resolve_group_column(train_raw, val_raw, test_raw)
assert_no_group_leakage(train_raw, val_raw, test_raw, GROUP_COL)
print(f"grupo de split/bootstrap: {GROUP_COL}")

count_rows = []
for scheme_id, scheme in LABEL_SCHEMES.items():
    tr, va, te = apply_label_scheme(train_raw, val_raw, test_raw, scheme, GROUP_COL)
    for split_name, frame in (("train", tr), ("val", va), ("test", te)):
        row = split_count_row(frame, split_name)
        row["experiment"] = scheme_id
        row["title"] = scheme["title"]
        row["n_groups"] = int(frame[GROUP_COL].nunique())
        count_rows.append(row)
        print(
            f"{scheme_id} {split_name}: pos={row['n_pos']} neg={row['n_neg']} "
            f"prev={row['prevalence']:.4f} groups={row['n_groups']} birads={row['birads']}"
        )

counts_preview = pd.DataFrame(count_rows)
counts_preview[["experiment", "split", "n", "n_pos", "n_neg", "prevalence", "n_groups"]]


## Métricas de reporte

Todo se calcula en el notebook a partir de las predicciones exportadas:

- ROC-AUC y PR-AUC en test
- baseline de PR-AUC = prevalencia positiva de **test**
- PR-AUC / baseline
- sensibilidad @ especificidad 90% y 95%
- precisión @ recall 80% (NaN si el modelo no alcanza ese recall)
- matriz de confusión en test con umbral Youden de **val**
- bootstrap **agrupado** por `study_id`/`patient_id`, 1000 repeticiones, IC 95%


In [ ]:
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def sensitivity_at_specificity(y_true, y_prob, target_spec: float) -> float:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    spec = 1.0 - fpr
    ok = spec >= target_spec
    if not ok.any():
        return float("nan")
    return float(tpr[ok].max())


def precision_at_recall(y_true, y_prob, target_recall: float) -> float:
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ok = recall >= target_recall
    if not ok.any():
        return float("nan")
    return float(precision[ok].max())


def grouped_bootstrap_auc(y_true, y_prob, groups, *, samples: int, seed: int) -> dict:
    y_true = np.asarray(y_true, dtype=np.int64).reshape(-1)
    y_prob = np.asarray(y_prob, dtype=np.float64).reshape(-1)
    groups = np.asarray(groups).reshape(-1)
    uniq = np.unique(groups)
    index_by_group = {group: np.flatnonzero(groups == group) for group in uniq}
    rng = np.random.default_rng(seed)
    roc_values = []
    pr_values = []
    attempts = 0
    max_attempts = samples * 20
    while len(roc_values) < samples and attempts < max_attempts:
        attempts += 1
        sampled = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([index_by_group[group] for group in sampled])
        yt = y_true[idx]
        yp = y_prob[idx]
        if np.unique(yt).size < 2:
            continue
        roc_values.append(roc_auc_score(yt, yp))
        pr_values.append(average_precision_score(yt, yp))
    if len(roc_values) < samples:
        raise RuntimeError(f"bootstrap agrupado: solo {len(roc_values)}/{samples} réplicas válidas")
    roc_values = np.asarray(roc_values, dtype=np.float64)
    pr_values = np.asarray(pr_values, dtype=np.float64)
    return {
        "test_roc_auc_ci_low": float(np.quantile(roc_values, 0.025)),
        "test_roc_auc_ci_high": float(np.quantile(roc_values, 0.975)),
        "test_pr_auc_ci_low": float(np.quantile(pr_values, 0.025)),
        "test_pr_auc_ci_high": float(np.quantile(pr_values, 0.975)),
        "bootstrap_valid_reps": int(len(roc_values)),
    }


def attach_groups(pred: pd.DataFrame, source: pd.DataFrame, group_col: str) -> pd.DataFrame:
    out = pred.copy()
    source = source.copy()
    keys = ["patient_id", "image_id"]
    for col in keys:
        out[col] = out[col].astype(str)
        source[col] = source[col].astype(str)
    extra_cols = [col for col in (group_col, "birads") if col in source.columns and col not in out.columns]
    if extra_cols:
        out = out.merge(source[keys + extra_cols].drop_duplicates(keys), on=keys, how="left")
    if group_col not in out.columns:
        out[group_col] = out["patient_id"].astype(str)
    return out


def metrics_from_predictions(val_pred, test_pred, *, group_col: str, seed: int, samples: int) -> dict:
    y_val_true = val_pred["y_true"].to_numpy()
    y_val_prob = val_pred["y_prob"].to_numpy()
    y_test_true = test_pred["y_true"].to_numpy()
    y_test_prob = test_pred["y_prob"].to_numpy()
    threshold = float(threshold_youden_j(y_val_true, y_val_prob))
    y_test_hat = apply_probability_threshold(y_test_prob, threshold)
    tn, fp, fn, tp = confusion_matrix(y_test_true, y_test_hat, labels=[0, 1]).ravel()
    n_pos = int((y_test_true == 1).sum())
    n_neg = int((y_test_true == 0).sum())
    prevalence = float(n_pos / (n_pos + n_neg))
    roc = float(roc_auc_score(y_test_true, y_test_prob))
    pr = float(average_precision_score(y_test_true, y_test_prob))
    boot = grouped_bootstrap_auc(
        y_test_true, y_test_prob, test_pred[group_col], samples=samples, seed=seed
    )
    return {
        "test_roc_auc": roc,
        "test_pr_auc": pr,
        "pr_auc_baseline": prevalence,
        "test_pr_auc_over_baseline": float(pr / prevalence) if prevalence else float("nan"),
        "test_sens_at_spec_90": sensitivity_at_specificity(y_test_true, y_test_prob, 0.90),
        "test_sens_at_spec_95": sensitivity_at_specificity(y_test_true, y_test_prob, 0.95),
        "test_prec_at_recall_80": precision_at_recall(y_test_true, y_test_prob, 0.80),
        "threshold_source": "youden_val",
        "threshold": threshold,
        "test_tn": int(tn),
        "test_fp": int(fp),
        "test_fn": int(fn),
        "test_tp": int(tp),
        "n_groups_test": int(pd.Series(test_pred[group_col]).nunique()),
        **boot,
    }


def counts_block(frame, prefix: str) -> dict:
    row = split_count_row(frame, prefix)
    return {
        f"{prefix}_n": row["n"],
        f"{prefix}_pos": row["n_pos"],
        f"{prefix}_neg": row["n_neg"],
        f"{prefix}_prevalence": row["prevalence"],
    }


def load_results_table() -> pd.DataFrame:
    if RESULTS_CSV.is_file():
        return pd.read_csv(RESULTS_CSV)
    return pd.DataFrame()


def upsert_result(row: dict) -> pd.DataFrame:
    table = load_results_table()
    incoming = pd.DataFrame([row])
    if not table.empty:
        table = table[table["experiment_id"].astype(str) != str(row["experiment_id"])]
        table = pd.concat([table, incoming], ignore_index=True)
    else:
        table = incoming
    order = {key: index for index, key in enumerate(LABEL_SCHEMES)}
    table["_ord"] = table["experiment_id"].map(order).fillna(99)
    table = table.sort_values(["_ord"]).drop(columns="_ord").reset_index(drop=True)
    table.to_csv(RESULTS_CSV, index=False)
    return table


def _fmt_cell(value) -> str:
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return "NA"
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.4f}"
    return str(value)


def dataframe_to_markdown(frame: pd.DataFrame, columns: list[str]) -> str:
    cols = [col for col in columns if col in frame.columns]
    header = "| " + " | ".join(cols) + " |"
    sep = "| " + " | ".join("---" for _ in cols) + " |"
    body = []
    for _, row in frame.iterrows():
        body.append("| " + " | ".join(_fmt_cell(row[col]) for col in cols) + " |")
    return "\n".join([header, sep, *body])


def write_markdown_report(table: pd.DataFrame) -> None:
    lines = [
        "# Ablación de etiquetas BI-RADS (VinDr-Mammo)",
        "",
        f"- Backbone: `resized_448_swint`",
        f"- Semilla: `{GENERAL['RANDOM_SEED']}`",
        f"- Grupo de split/bootstrap: `{GROUP_COL}`",
        f"- Bootstrap: {BOOTSTRAP_REPS} repeticiones, IC 95%, agrupado por `{GROUP_COL}`",
        f"- Umbral: Youden en validación (nunca test)",
        "",
        "## Conteos y prevalencia",
        "",
    ]
    if table.empty:
        lines.append("Todavía no hay experimentos corridos.")
    else:
        count_cols = [
            "experiment_id",
            "title",
            "train_pos",
            "train_neg",
            "train_prevalence",
            "val_pos",
            "val_neg",
            "val_prevalence",
            "test_pos",
            "test_neg",
            "test_prevalence",
        ]
        lines.append(dataframe_to_markdown(table, count_cols))
        lines.extend(["", "## Métricas en test", ""])
        metric_cols = [
            "experiment_id",
            "test_roc_auc",
            "test_roc_auc_ci_low",
            "test_roc_auc_ci_high",
            "test_pr_auc",
            "test_pr_auc_ci_low",
            "test_pr_auc_ci_high",
            "pr_auc_baseline",
            "test_pr_auc_over_baseline",
            "test_sens_at_spec_90",
            "test_sens_at_spec_95",
            "test_prec_at_recall_80",
            "threshold",
            "test_tn",
            "test_fp",
            "test_fn",
            "test_tp",
        ]
        lines.append(dataframe_to_markdown(table, metric_cols))
        lines.extend(["", "## Comet", ""])
        lines.append(dataframe_to_markdown(table, ["experiment_id", "comet_name", "comet_url"]))

    by_id = (
        {str(row["experiment_id"]): row.to_dict() for _, row in table.iterrows()}
        if not table.empty
        else {}
    )
    a, b, c = by_id.get("A"), by_id.get("B"), by_id.get("C")
    lines.extend(["", "## Interpretación", ""])
    if a is not None and b is not None:
        d_roc = float(b["test_roc_auc"]) - float(a["test_roc_auc"])
        d_pr = float(b["test_pr_auc"]) - float(a["test_pr_auc"])
        lines.append(
            f"- **B vs A** (¿BI-RADS 3 baja el techo?): ΔROC-AUC={d_roc:+.4f}, ΔPR-AUC={d_pr:+.4f}."
        )
        if d_roc >= 0.03 or d_pr >= 0.05:
            lines.append("  - B mejora de forma material → BI-RADS 3 está mezclando señal y baja el techo de A.")
        else:
            lines.append("  - B no se separa mucho de A → excluir BI-RADS 3 no basta para explicar el techo.")
    else:
        lines.append("- Falta A o B para comparar `1-2 vs 3-5` contra `1-2 vs 4-5`.")
    if a is not None and c is not None:
        d_roc = float(c["test_roc_auc"]) - float(a["test_roc_auc"])
        prev_a = float(a["test_prevalence"])
        prev_c = float(c["test_prevalence"])
        lines.append(
            f"- **C vs A** (BI-RADS 3 como negativo): ΔROC-AUC={d_roc:+.4f}; "
            f"prevalencia test {prev_a:.4f} → {prev_c:.4f}; "
            f"PR-AUC/baseline A={float(a['test_pr_auc_over_baseline']):.3f}, "
            f"C={float(c['test_pr_auc_over_baseline']):.3f}."
        )
        if abs(prev_c - prev_a) / max(prev_a, 1e-9) >= 0.25:
            lines.append("  - La prevalencia cambió mucho; comparar PR-AUC crudo es engañoso, usar PR-AUC/baseline.")
    else:
        lines.append("- Falta A o C para comparar `1-3 vs 4-5` contra el setup actual.")
    if a is not None and b is not None and c is not None:
        rocs = [float(a["test_roc_auc"]), float(b["test_roc_auc"]), float(c["test_roc_auc"])]
        if max(rocs) - min(rocs) < 0.03 and abs(sum(rocs) / 3 - 0.75) < 0.05:
            lines.append(
                "- Los tres experimentos siguen cerca de ROC-AUC ~0.75 → el cuello probablemente no es BI-RADS 3 "
                "(señal visual, domain shift, resolución, arquitectura o formulación por vista/estudio)."
            )
    missing = [key for key in LABEL_SCHEMES if key not in by_id]
    if missing:
        lines.append(f"- Pendientes de entrenar: {', '.join(missing)}.")
    lines.append("")
    RESULTS_MD.write_text("\n".join(lines), encoding="utf-8")
    print(f"Escrito {RESULTS_MD}")


print("helpers de métricas listos")


## Cómo correr cada experimento

1. Ejecutar las celdas de setup, CONFIG, dataset y helpers (una vez por sesión).
2. En la celda siguiente, setear `RUN_EXPERIMENTS` y correrla.
3. Correr la celda de reporte para regenerar CSV + Markdown.

```python
RUN_EXPERIMENTS = ["A"]          # baseline actual 1-2 vs 3-5
RUN_EXPERIMENTS = ["B"]          # excluye BI-RADS 3
RUN_EXPERIMENTS = ["C"]          # BI-RADS 3 como negativo
RUN_EXPERIMENTS = ["A", "B", "C"]  # los tres (3 entrenamientos completos)
```

`FORCE_RERUN = False` saltea un ID que ya esté en `results/birads_label_ablation.csv`. Ponelo en `True` para reentrenar.

Nombres Comet: `wip_a_12_vs_35_resized_448_swint`, `wip_b_12_vs_45_resized_448_swint`, `wip_c_123_vs_45_resized_448_swint`.


In [ ]:
# Elegir qué entrenar en esta sesión:
RUN_EXPERIMENTS = ["A"]  # ["B"] | ["C"] | ["A", "B", "C"]
FORCE_RERUN = False

existing = load_results_table()
done_ids = set(existing["experiment_id"].astype(str)) if not existing.empty else set()
print("ya en CSV:", sorted(done_ids) or "nada")

summaries = []
for scheme_id in RUN_EXPERIMENTS:
    scheme = LABEL_SCHEMES[scheme_id]
    prefix = comet_prefix(scheme_id)
    if scheme_id in done_ids and not FORCE_RERUN:
        print(f"salteo {scheme_id} ({prefix}): ya está en {RESULTS_CSV}")
        continue

    train_s, val_s, test_s = apply_label_scheme(train_raw, val_raw, test_raw, scheme, GROUP_COL)
    CONFIG["GENERAL"]["BIRADS_LABEL_SCHEME"] = scheme_id
    CONFIG["GENERAL"]["BIRADS_NEGATIVE"] = list(scheme["negative"])
    CONFIG["GENERAL"]["BIRADS_POSITIVE"] = list(scheme["positive"])
    CONFIG["GENERAL"]["BIRADS_DROP"] = list(scheme["drop"])
    CONFIG["GENERAL"]["WIP_EXPERIMENT"] = prefix
    set_random_seeds(CONFIG)

    print(f"\n=== {scheme_id} {scheme['title']}  comet={prefix}_resized_448_swint ===")
    for split_name, frame in (("train", train_s), ("val", val_s), ("test", test_s)):
        row = split_count_row(frame, split_name)
        print(f"  {split_name}: pos={row['n_pos']} neg={row['n_neg']} prev={row['prevalence']:.4f}")

    summary = run_training_experiment(
        CONFIG,
        "resized",
        BACKBONE_NAME,
        train_s,
        val_s,
        test_s,
        input_size=INPUT_SIZE,
        experiment_suffix=prefix,
        return_summary=True,
    )
    if not summary.get("val_predictions_file") or not summary.get("test_predictions_file"):
        raise RuntimeError("Hace falta CONFIG['EVALUATION']['EXPORT_PREDICTIONS']=True para el reporte")
    val_pred = attach_groups(pd.read_csv(summary["val_predictions_file"]), val_s, GROUP_COL)
    test_pred = attach_groups(pd.read_csv(summary["test_predictions_file"]), test_s, GROUP_COL)
    extra = metrics_from_predictions(
        val_pred,
        test_pred,
        group_col=GROUP_COL,
        seed=int(GENERAL["RANDOM_SEED"]),
        samples=BOOTSTRAP_REPS,
    )
    result = {
        "experiment_id": scheme_id,
        "scheme_key": scheme["key"],
        "title": scheme["title"],
        "comet_name": summary.get("experiment", f"{prefix}_resized_448_swint"),
        "comet_url": summary.get("comet_url"),
        "negative_birads": "-".join(str(v) for v in scheme["negative"]),
        "positive_birads": "-".join(str(v) for v in scheme["positive"]),
        "dropped_birads": "-".join(str(v) for v in scheme["drop"]) or "",
        "backbone": BACKBONE_NAME,
        "mode": "resized",
        "input_size": "448x448",
        "group_column": GROUP_COL,
        "bootstrap_n": BOOTSTRAP_REPS,
        "val_best_metric": summary.get("val_best_metric"),
        "val_best_metric_name": summary.get("val_best_metric_name"),
        "train_predictions_file": summary.get("train_predictions_file"),
        "val_predictions_file": summary.get("val_predictions_file"),
        "test_predictions_file": summary.get("test_predictions_file"),
        **counts_block(train_s, "train"),
        **counts_block(val_s, "val"),
        **counts_block(test_s, "test"),
        **extra,
    }
    summaries.append(result)
    table = upsert_result(result)
    try:
        write_markdown_report(table)
    except Exception as exc:
        print(f"[aviso] el CSV se guardó pero falló el Markdown: {exc}")
    print(pd.Series(result)[["experiment_id", "test_roc_auc", "test_pr_auc", "pr_auc_baseline", "test_pr_auc_over_baseline"]])

print("entrenamientos de esta celda:", [row["experiment_id"] for row in summaries] or "ninguno (todo salteado)")


## Resultados agregados

Lee `results/birads_label_ablation.csv` (incluye corridas de sesiones previas) y reescribe el Markdown. Correr esta celda después de cada experimento.


In [ ]:
from IPython.display import display

table = load_results_table()
write_markdown_report(table)
if table.empty:
    print("CSV vacío: todavía no hay corridas. Ejecutá la celda de entrenamiento.")
else:
    display_cols = [
        "experiment_id",
        "title",
        "train_pos",
        "train_neg",
        "test_pos",
        "test_neg",
        "test_prevalence",
        "test_roc_auc",
        "test_roc_auc_ci_low",
        "test_roc_auc_ci_high",
        "test_pr_auc",
        "test_pr_auc_ci_low",
        "test_pr_auc_ci_high",
        "pr_auc_baseline",
        "test_pr_auc_over_baseline",
        "test_sens_at_spec_90",
        "test_sens_at_spec_95",
        "test_prec_at_recall_80",
        "threshold",
        "test_tn",
        "test_fp",
        "test_fn",
        "test_tp",
        "comet_name",
    ]
    display_cols = [col for col in display_cols if col in table.columns]
    display(table[display_cols])
    print(f"CSV: {RESULTS_CSV.resolve()}")
    print(f"MD:  {RESULTS_MD.resolve()}")
    print(RESULTS_MD.read_text(encoding="utf-8"))


## Notas de lectura (después de A, B y C)

- PR-AUC crudo **no** se compara entre esquemas si la prevalencia de test cambia (sobre todo A vs C). Usar `PR-AUC / baseline`.
- Sensibilidad a spec 90/95 y precisión a recall 80% son independientes del umbral Youden; la matriz de confusión sí usa ese umbral de val.
- El undersample 3:1 del JSON canónico se calculó con las labels de A. B/C reetiquetan esas mismas filas de train (B además saca BI-RADS 3). Así A replica el setup actual; B y C aíslan la definición del label, no un split nuevo.
